# 🤖 BridgeAI — Arabic ↔ English Communication Assistant
## Day 1 & 2: Translation Pipeline + Sentiment Detection

**Author:** Vikas Varma Mudunuru  
**MSc Data Science & AI — Middlesex University Dubai**

### What this notebook does:
- Translates text between Arabic and English automatically
- Detects whether input is Arabic or English (auto-routing)
- Analyses the sentiment (positive/negative/neutral) in both languages
- Combines everything into one unified pipeline function

### Models used:
| Task | Model | Source |
|---|---|---|
| Arabic → English Translation | Helsinki-NLP/opus-mt-ar-en | HuggingFace |
| English → Arabic Translation | Helsinki-NLP/opus-mt-en-ar | HuggingFace |
| English Sentiment | distilbert-base-uncased-finetuned-sst-2-english | HuggingFace |
| Arabic Sentiment | CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment | HuggingFace |

---
## 📦 Cell 1 — Install Required Libraries

In [1]:
# The ! means run this as a terminal command inside Colab, not as Python code
# transformers  → HuggingFace library that gives us access to 1000s of pretrained AI models
# torch         → PyTorch, the deep learning engine that runs the models under the hood
# langdetect    → detects what language a piece of text is written in (returns 'ar', 'en' etc)
# sentencepiece → tokenizer library required by some translation models to split text into pieces
# -q            → quiet mode, suppresses long installation logs

!pip install transformers torch langdetect sentencepiece -q

print("All libraries installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 12.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
All libraries installed!


---
## 🌐 Cell 2 — Load Translation Models

In [2]:
# MarianMTModel   → the actual neural network that does the translation
#                   'Marian' is a popular open-source machine translation system
#                   developed by Helsinki University, trained on millions of sentence pairs
# MarianTokenizer → converts raw text into numbers the model can understand
#                   (models can't read words directly, only numbers)
from transformers import MarianMTModel, MarianTokenizer

# This is the model name/address on HuggingFace's model hub
# 'ar-en' means Arabic to English direction
# 'en-ar' means English to Arabic direction
# These are TWO separate models because translation is not just reversing words
ar_en_model_name = "Helsinki-NLP/opus-mt-ar-en"
en_ar_model_name = "Helsinki-NLP/opus-mt-en-ar"

# from_pretrained downloads the model weights from HuggingFace and loads into memory
# We always load BOTH tokenizer and model separately:
#   tokenizer → handles text→numbers→text conversion
#   model     → does the actual translation using those numbers
print("Loading Arabic → English model... (may take 1-2 mins)")
ar_en_tokenizer = MarianTokenizer.from_pretrained(ar_en_model_name)
ar_en_model = MarianMTModel.from_pretrained(ar_en_model_name)

print("Loading English → Arabic model... (may take 1-2 mins)")
en_ar_tokenizer = MarianTokenizer.from_pretrained(en_ar_model_name)
en_ar_model = MarianMTModel.from_pretrained(en_ar_model_name)

print("\nBoth translation models loaded successfully!")

Loading Arabic → English model... (may take 1-2 mins)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/917k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]


Both translation models loaded successfully!


---
## 🔄 Cell 3 — Translation Function

In [3]:
# Import the detect function from langdetect library
# detect('hello') returns 'en', detect('مرحبا') returns 'ar'
from langdetect import detect

def translate(text):
    """
    Translates text between Arabic and English automatically.
    Auto-detects the input language and routes to correct model.

    Args:
        text (str): Input text in either Arabic or English

    Returns:
        dict: Contains original, translated, direction, detected_language
    """

    # Try to detect the language of input text
    # try/except is a safety net — if detection fails (text too short, symbols only)
    # we default to English instead of crashing the whole app
    try:
        lang = detect(text)
    except:
        lang = "en"  # default to English if detection fails

    if lang == "ar":
        # Arabic detected → use Arabic to English model

        # Tokenize: convert text into PyTorch tensors (numbers)
        # return_tensors="pt" means PyTorch format
        # padding=True ensures consistent input sizes (models need fixed dimensions)
        inputs = ar_en_tokenizer(text, return_tensors="pt", padding=True)

        # generate() is where the actual AI translation happens
        # **inputs unpacks the dictionary into keyword arguments (common Python pattern)
        outputs = ar_en_model.generate(**inputs)

        # Decode: convert output numbers back into readable text
        # outputs[0] gets the first translation result
        # skip_special_tokens=True removes internal model tokens like <pad> </s>
        translated = ar_en_tokenizer.decode(outputs[0], skip_special_tokens=True)
        direction = "Arabic → English"

    else:
        # English (or any non-Arabic language) → use English to Arabic model
        inputs = en_ar_tokenizer(text, return_tensors="pt", padding=True)
        outputs = en_ar_model.generate(**inputs)
        translated = en_ar_tokenizer.decode(outputs[0], skip_special_tokens=True)
        direction = "English → Arabic"

    # Return a dictionary with all useful information
    # Returning a dict (not just a string) makes it easy to access
    # different parts of the result in the UI later
    return {
        "original": text,
        "translated": translated,
        "direction": direction,
        "detected_language": lang
    }

print("translate() function defined and ready!")

translate() function defined and ready!


---
## 🧪 Cell 4 — Test Translation Pipeline

In [4]:
# Test 1: English → Arabic
# f"" is an f-string — embeds variables directly inside strings using {}
test1 = translate("Hello, how are you?")
print(f"Direction:  {test1['direction']}")
print(f"Original:   {test1['original']}")
print(f"Translated: {test1['translated']}")
print()

# Test 2: Arabic → English
test2 = translate("مرحبا، كيف حالك؟")
print(f"Direction:  {test2['direction']}")
print(f"Original:   {test2['original']}")
print(f"Translated: {test2['translated']}")
print()

# Batch test: mix of Arabic and English phrases
# This proves auto-detection works without us manually specifying the language
phrases = [
    "Good morning, welcome to Dubai!",
    "أنا سعيد جداً بلقائك",           # I am very happy to meet you
    "The weather is beautiful today",
    "شكراً جزيلاً على مساعدتك"        # Thank you so much for your help
]

print("--- Batch Translation Test ---")
for phrase in phrases:
    result = translate(phrase)
    print(f"{result['direction']}: {result['original']}")
    print(f"→ {result['translated']}")
    print()

Direction:  English → Arabic
Original:   Hello, how are you?
Translated: مرحباً، كيف حالك؟

Direction:  Arabic → English
Original:   مرحبا، كيف حالك؟
Translated: Hey, how are you?

--- Batch Translation Test ---
English → Arabic: Good morning, welcome to Dubai!
→ صباح الخير، مرحبا بكم في دبي!

Arabic → English: أنا سعيد جداً بلقائك
→ I'm so happy to meet you.

English → Arabic: The weather is beautiful today
→ الطقس جميل اليوم

Arabic → English: شكراً جزيلاً على مساعدتك
→ Thank you so much for your help.



---
## 😊 Cell 5 — Load Sentiment Analysis Models

In [5]:
# pipeline() is HuggingFace's shortcut that bundles tokenizer + model + post-processing
# into ONE simple function call — we don't need to manually tokenize and decode
# We don't train these models — they're already trained by researchers
# on millions of Arabic and English texts. We just load and use them.
from transformers import pipeline

# English sentiment model:
# distilbert = smaller, faster version of BERT (65% smaller, 97% as accurate)
# finetuned-sst-2 = fine-tuned on Stanford Sentiment Treebank dataset
# Returns: POSITIVE or NEGATIVE with a confidence score
print("Loading English sentiment model...")
english_sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Arabic sentiment model:
# CAMeL-Lab = Center for Arabic Language Processing at NYU Abu Dhabi
# camelbert-da = trained on dialectal Arabic (Gulf, Egyptian, Levantine etc)
# This is important for UAE — people write in Gulf dialect not just formal Arabic
print("Loading Arabic sentiment model...")
arabic_sentiment = pipeline(
    "sentiment-analysis",
    model="CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment"
)

print("\nBoth sentiment models loaded!")

Loading English sentiment model...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

BertForSequenceClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


Both sentiment models loaded!


---
## 🎭 Cell 6 — Sentiment Detection Function

In [6]:
def get_sentiment(text):
    """
    Detects sentiment (positive/negative/neutral) of input text.
    Auto-detects language and routes to correct sentiment model.

    Args:
        text (str): Input text in Arabic or English

    Returns:
        dict: Contains text, language, sentiment label, confidence score
    """

    # Detect language — same pattern as translate() function
    try:
        lang = detect(text)
    except:
        lang = "en"

    # Route to correct model based on detected language
    # [0] gets the first result — pipeline returns a list, we only need one
    if lang == "ar":
        result = arabic_sentiment(text)[0]
    else:
        result = english_sentiment(text)[0]

    # Normalise labels to consistent format
    # Different models return different label names:
    # English model returns: 'POSITIVE' or 'NEGATIVE'
    # Arabic model returns:  'LABEL_0', 'LABEL_1', 'POS', 'NEG'
    # We normalise all of these into one consistent format with emojis
    label = result["label"].upper()

    if label in ["LABEL_0", "NEGATIVE", "NEG"]:
        label = "NEGATIVE 😞"
    elif label in ["LABEL_1", "POSITIVE", "POS"]:
        label = "POSITIVE 😊"
    else:
        label = "NEUTRAL 😐"

    # result["score"] is a decimal between 0 and 1 (e.g. 0.986)
    # multiply by 100 and round to 1 decimal to get percentage (e.g. 98.6%)
    confidence = round(result["score"] * 100, 1)

    return {
        "text": text,
        "language": lang,
        "sentiment": label,
        "confidence": confidence
    }

print("get_sentiment() function defined and ready!")

get_sentiment() function defined and ready!


---
## 🧪 Cell 7 — Test Sentiment Detection

In [7]:
# Test phrases — mix of positive, negative, neutral in both languages
test_phrases = [
    "I love Dubai so much, it's amazing!",      # clearly positive English
    "This service is terrible, I'm very angry", # clearly negative English
    "The weather is okay today",                # neutral English
    "أنا سعيد جداً بهذا المكان",               # I am very happy with this place
    "هذا الطعام سيء جداً",                     # This food is very bad
    "الجو معتدل اليوم"                          # The weather is moderate today
]

print("--- Sentiment Analysis Test ---\n")
for phrase in test_phrases:
    result = get_sentiment(phrase)

    # Show flag emoji based on detected language for visual clarity
    lang_label = "🇬🇧 English" if result["language"] == "en" else "🇦🇪 Arabic"

    print(f"{lang_label}: {result['text']}")
    print(f"Sentiment: {result['sentiment']} | Confidence: {result['confidence']}%")
    print()

--- Sentiment Analysis Test ---

🇬🇧 English: I love Dubai so much, it's amazing!
Sentiment: POSITIVE 😊 | Confidence: 100.0%

🇬🇧 English: This service is terrible, I'm very angry
Sentiment: NEGATIVE 😞 | Confidence: 100.0%

🇬🇧 English: The weather is okay today
Sentiment: POSITIVE 😊 | Confidence: 100.0%

🇦🇪 Arabic: أنا سعيد جداً بهذا المكان
Sentiment: POSITIVE 😊 | Confidence: 98.6%

🇦🇪 Arabic: هذا الطعام سيء جداً
Sentiment: NEGATIVE 😞 | Confidence: 99.4%

🇦🇪 Arabic: الجو معتدل اليوم
Sentiment: NEGATIVE 😞 | Confidence: 63.3%



---
## 🚀 Cell 8 — Full BridgeAI Pipeline (Translation + Sentiment Combined)

In [8]:
def analyze(text):
    """
    Master function that runs the complete BridgeAI pipeline.
    Combines translation + sentiment detection in one call.
    This is the function the FastAPI backend will expose as an API endpoint.

    Args:
        text (str): Any text in Arabic or English

    Returns:
        dict: Full analysis including translation, direction,
              sentiment of original AND translated text
    """

    # Step 1: Detect language
    try:
        lang = detect(text)
    except:
        lang = "en"

    # Step 2: Translate the text
    # This calls our translate() function defined in Cell 3
    translation = translate(text)

    # Step 3: Get sentiment of the ORIGINAL text
    # This calls our get_sentiment() function defined in Cell 6
    sentiment_original = get_sentiment(text)

    # Step 4: Get sentiment of the TRANSLATED text
    # We check sentiment in both languages to ensure consistency
    # If Arabic text is NEGATIVE, the English translation should also be NEGATIVE
    # This cross-language sentiment check is what makes BridgeAI smart
    sentiment_translated = get_sentiment(translation["translated"])

    # Return everything in one clean dictionary
    return {
        "original": text,
        "translated": translation["translated"],
        "direction": translation["direction"],
        "detected_language": lang,
        "sentiment_original": sentiment_original["sentiment"],
        "sentiment_original_confidence": sentiment_original["confidence"],
        "sentiment_translated": sentiment_translated["sentiment"],
        "sentiment_translated_confidence": sentiment_translated["confidence"],
    }

print("analyze() master function defined and ready!")

analyze() master function defined and ready!


---
## 🎯 Cell 9 — Final Pipeline Test

In [9]:
# Final end-to-end test of the complete BridgeAI pipeline
# These test cases cover: positive English, negative Arabic, positive Arabic
test_inputs = [
    "I'm really excited to visit Dubai next week!",
    "أنا غاضب جداً من هذه الخدمة السيئة",  # I am very angry about this bad service
    "الحمد لله، كل شيء على ما يرام"         # Thank God, everything is fine
]

print("=" * 60)
print("BRIDGEAI FULL PIPELINE TEST")
print("=" * 60)
print()

for text in test_inputs:
    # Call the master analyze() function — does everything in one shot
    result = analyze(text)

    print(f"INPUT:       {result['original']}")
    print(f"TRANSLATION: {result['translated']}")
    print(f"DIRECTION:   {result['direction']}")
    print(f"SENTIMENT (original):   {result['sentiment_original']} ({result['sentiment_original_confidence']}%)")
    print(f"SENTIMENT (translated): {result['sentiment_translated']} ({result['sentiment_translated_confidence']}%)")
    print("-" * 60)

BRIDGEAI FULL PIPELINE TEST

INPUT:       I'm really excited to visit Dubai next week!
TRANSLATION: أنا متحمسة حقاً لزيارة (دبي) الأسبوع القادم!
DIRECTION:   English → Arabic
SENTIMENT (original):   POSITIVE 😊 (100.0%)
SENTIMENT (translated): POSITIVE 😊 (96.7%)
------------------------------------------------------------
INPUT:       أنا غاضب جداً من هذه الخدمة السيئة
TRANSLATION: I'm so mad at this shitty service.
DIRECTION:   Arabic → English
SENTIMENT (original):   NEGATIVE 😞 (98.7%)
SENTIMENT (translated): NEGATIVE 😞 (100.0%)
------------------------------------------------------------
INPUT:       الحمد لله، كل شيء على ما يرام
TRANSLATION: Thank God. Everything's fine.
DIRECTION:   Arabic → English
SENTIMENT (original):   POSITIVE 😊 (97.0%)
SENTIMENT (translated): POSITIVE 😊 (100.0%)
------------------------------------------------------------


---
## 📋 Summary

### What we built in Day 1 & 2:

| Component | Function | Status |
|---|---|---|
| Language Detection | `detect()` from langdetect | ✅ Done |
| Arabic → English Translation | `translate()` using Helsinki-NLP | ✅ Done |
| English → Arabic Translation | `translate()` using Helsinki-NLP | ✅ Done |
| English Sentiment | `get_sentiment()` using DistilBERT | ✅ Done |
| Arabic Sentiment | `get_sentiment()` using CAMeL-BERT | ✅ Done |
| Full Pipeline | `analyze()` master function | ✅ Done |

### Coming next:
- **Day 3:** Wrap everything in a FastAPI backend (real web API)
- **Day 4:** Build the frontend chat UI (HTML/CSS/JS)
- **Day 5:** Add voice input + text-to-speech output (the Jarvis moment 🎤)
- **Day 6:** Cultural context tips + UI polish
- **Day 7:** Deploy live + README + GitHub